## The Gateway API — Ingress, but more

`Ingress` is showing its age — half of every controller's behaviour lives in **annotations** (controller-specific magic strings), because the resource lacks fields for rate-limiting, header rewrites, or traffic splitting. The **Gateway API** is the official successor. It splits the one `Ingress` object into three, matching who owns what:

- **`GatewayClass`** — a kind of gateway implementation (the `IngressClass` analogue).
- **`Gateway`** — a *physical* listener: IP, port, TLS. Owned by platform/infra teams.
- **`HTTPRoute`** (+ `GRPCRoute`, `TCPRoute`, `TLSRoute`) — routing rules attached to a Gateway. Owned by app teams.

```yaml
kind: Gateway
spec:
  gatewayClassName: nginx
  listeners:
    - { name: https, port: 443, protocol: HTTPS, tls: { certificateRefs: [{ name: web-tls }] } }
---
kind: HTTPRoute
spec:
  parentRefs: [{ name: public }]
  hostnames: ["hello.example.com"]
  rules:
    - matches: [{ path: { type: PathPrefix, value: / } }]
      backendRefs: [{ name: hello, port: 80 }]
```

Why bother: **cleaner role split** (platform owns Gateways, apps own Routes — RBAC matches), **richer matching** (headers, query params, methods), **first-class traffic splitting** (90% v1 / 10% v2 as a field, not an annotation), and **protocol diversity** (TCP/UDP/gRPC/TLS).

**Status 2026:** Gateway API v1 is GA and most controllers support it alongside Ingress. **The CKA still tests `Ingress`** — learn it first; expect Gateways in production within a year or two. On our map the Gateway API is the next generation of the same **Ingress** chip — same L7 job, cleaner model.